In [1]:
%%time
! pip install --upgrade gradio
!pip install -q langchain langchain-openai faiss-cpu pypdf
!pip install langchain langchain-community
!pip install langchain langchain-huggingface
! pip install bs4
! pip install langchain
! pip install langchain-core
! pip install openai
! pip install -q google-genai openai
! pip install -q langgraph google-genai
! pip install sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.3/73.3 kB 5.7 MB/s eta 0:00:00
  Attempting uninstall: starlette
    Found existing installation: starlette 0.52.1
    Uninstalling starlette-0.52.1:
      Successfully uninstalled starlette-0.52.1
  Attempting uninstall: gradio-client
    Found existing installation: gradio_client 1.14.0
    Uninstalling gradio_client-1.14.0:
      Successfully uninstalled gradio_client-1.14.0
  Attempting uninstall: gradio
    Found existing installation: gradio 5.50.0
    Uninstalling gradio-5.50.0:
      Successfully uninstalled gradio-5.50.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires starlette<1.0.0,>=0.49.1, but you have starlette 1.2.1 whi

In [2]:
%%time
import gradio as gr
import getpass
import os
import bs4
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from openai import OpenAI
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing_extensions import List, TypedDict
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import ChatPromptTemplate
from google import genai
from google.genai import types
from google.colab import drive

<timed exec>:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.


CPU times: user 31 s, sys: 3.95 s, total: 34.9 s
Wall time: 46.4 s


# Import Dataset from Google Drive

In [3]:
drive.mount('/content/drive')

dataset = '/content/drive/MyDrive/corpus_first_100.csv'



Mounted at /content/drive


# Load Dataset

In [4]:
import pandas as pd
from langchain_core.documents import Document # Import Document class

loader_df = pd.read_csv(dataset)

# Convert DataFrame rows to Document objects
docs = []
for index, row in loader_df.iterrows():
    docs.append(Document(page_content=row['text']))

# Join the page content from the loaded documents into a single string for test_chatbot
financial_chatbot = "\n\n".join([doc.page_content for doc in docs])

In [5]:
financial_chatbot

'I\'m not saying I don\'t like the idea of on-the-job training too, but you can\'t expect the company to do that. Training workers is not their job - they\'re building software. Perhaps educational systems in the U.S. (or their students) should worry a little about getting marketable skills in exchange for their massive investment in education, rather than getting out with thousands in student debt and then complaining that they aren\'t qualified to do anything.\n\nSo nothing preventing false ratings besides additional scrutiny from the market/investors, but there are some newer controls in place to prevent institutions from using them. Under the DFA banks can no longer solely rely on credit ratings as due diligence to buy a financial instrument, so that\'s a plus. The intent being that if financial institutions do their own leg work then *maybe* they\'ll figure out that a certain CDO is garbage or not. Edit: lead in\n\nYou can never use a health FSA for individual health insurance pre

# Setup Embedding

In [6]:
# 2. Setting up Embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

print("Setup complete. Your chatbot is ready for testing, Let's Go Isaac!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Setup complete. Your chatbot is ready for testing, Let's Go Isaac!


# Open AI GPT 4 model

In [7]:
%%time
client = OpenAI(
  api_key=""
)
response = client.responses.create(
  model="gpt-4-turbo",
  input="I forgive you",
  store=True,
)

print(response.output_text);

Thank you for your forgiveness. If there's anything more you'd like to discuss or any questions you have, feel free to let me know. I'm here to help!
CPU times: user 1.9 s, sys: 138 ms, total: 2.04 s
Wall time: 7.56 s


# Vector Embedding

In [8]:
%%time
import os

# Set the OpenAI API key as an environment variable
os.environ["OPENAI_API_KEY"] = ""

llm = ChatOpenAI(model="gpt-4-turbo", temperature=0.8, max_tokens=250)

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

CPU times: user 182 ms, sys: 4.91 ms, total: 186 ms
Wall time: 186 ms


In [9]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

# Chunking

In [10]:
%%time
text_splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=50)
all_splits = text_splitter.split_documents(docs)

CPU times: user 20.8 ms, sys: 0 ns, total: 20.8 ms
Wall time: 21.9 ms


In [11]:
%%time
# Create FAISS index from chunks
db = FAISS.from_documents(all_splits, embeddings)

CPU times: user 677 ms, sys: 80.1 ms, total: 757 ms
Wall time: 6.38 s


# Prompting instructions

In [12]:
chatbot_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a financial analyst at a stock firm. You need to answer questions related to some financial review in a  polite manner to a prospective client.

         Follow these rules:
1. Always answer based on the provided context only. Do NOT make up an answer.
2. If unsure, say "I don't have enough information to answer that"
3. Dealing with strict confidential information divulge only information asked
4. Use bullet points if needed

Context:
{context}"""),
    ("human", "Question: {question}"),
    ("ai", "Answer: "),
])

def ask_question(question: str) -> str:
    # Retrieve context, explicitly setting k to limit documents
    retrieved_docs = db.similarity_search(question, k=4)

    # Format documents
    docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)

    '''# Print the retrieved context for debugging
    print("--- Retrieved Context ---")
    print(docs_content)
    print("-------------------------")'''

    # Create prompt using your custom template
    messages = chatbot_prompt.format_messages(
        question=question,
        context=docs_content
    )

    # Get LLM response
    response = llm.invoke(messages)

    return response.content

In [13]:
%%time
first_question = ask_question("What is a 401k?")
first_question

CPU times: user 58.1 ms, sys: 8.01 ms, total: 66.1 ms
Wall time: 5.88 s


'A 401(k) plan is a tax-advantaged retirement savings plan sponsored by an employer. It allows workers to save and invest a portion of their paycheck before taxes are taken out. Taxes aren’t paid until the money is withdrawn from the account. 401(k) plans are a common form of defined contribution retirement plan.'

In [14]:
second_question = ask_question("What is a premium?")
second_question

'A premium in the context of insurance is the amount paid by the insured to the insurance company to cover the risk and provide insurance protection for a specified period. In the example provided, the insurance premium is $150, which ensures coverage for potential damage or loss, such as an accident involving the boat, with a compensation value up to $10,000.'

In [15]:
third_question = ask_question("What is a bond?")
third_question

"A bond is a fixed income instrument that represents a loan made by an investor to a borrower (typically corporate or governmental). Here are some key points about bonds:\n\n- **Issuer and Investor**: A bond is issued by corporations, municipalities, states, and sovereign governments to finance projects and operations.\n- **Principal and Interest**: The issuer of the bond owes the holders a debt and is obliged to pay them interest (the coupon) and/or to repay the principal at a later date, termed the maturity date.\n- **Income**: Bonds typically provide periodic payments of interest and return the principal at maturity.\n- **Risk and Creditworthiness**: The creditworthiness of the issuer can affect a bond's price and its yield. Higher risk generally leads to higher potential returns.\n- **Types**: There are various types of bonds including U.S. Treasury bonds, municipal bonds, corporate bonds, and others, each with unique characteristics, tax implications, and risk levels.\n\nBonds are

In [16]:
fourth_question = ask_question("What is a stock?")
fourth_question

"A stock represents ownership in a corporation and signifies a claim on part of the corporation's assets and earnings. Here are some key points about stocks:\n\n- **Ownership**: When you buy a stock, you become one of the owners of the company, known as a shareholder. This entitles you to vote at shareholders' meetings and receive dividends if they are distributed.\n- **Types**: There are mainly two types of stocks: common and preferred. Common stock usually entitles the owner to vote at shareholders' meetings and to receive dividends. Preferred stock generally does not provide voting rights but has a higher claim on assets and earnings than the common shares.\n- **Markets**: Stocks are bought and sold on stock exchanges, like the New York Stock Exchange (NYSE) or the NASDAQ. The price of a stock fluctuates based on supply and demand in the market, influenced by the company’s performance, investor perceptions, and general market conditions. \n\nOwning stocks is one way investors can bu

In [17]:
fifth_question = ask_question("What is the difference between a bond and a stock")
fifth_question

"I don't have enough information to answer that based on the context provided. However, I can help clarify the specific characteristics of warrants, preferred stock, and Treasury Bonds as mentioned in the context if that would be helpful."

In [18]:
sixth_question = ask_question("What are the metrics used to evaluate a stock?")
sixth_question

"Certainly! Here are some key metrics typically used to evaluate a stock:\n\n- **Earnings Per Share (EPS)**: This measures the profitability of a company on a per-share basis, helping investors understand how much money the company makes for each share of its stock.\n\n- **Price-to-Earnings Ratio (P/E)**: This ratio compares a company's stock price to its earnings per share, providing a sense of the stock's valuation relative to its earnings.\n\n- **Dividend Yield**: This is a financial ratio that shows how much a company pays out in dividends each year relative to its stock price. It is often used by investors seeking income from their investments.\n\n- **Return on Equity (ROE)**: This measures a corporation's profitability by revealing how much profit a company generates with the money shareholders have invested. It is expressed as a percentage.\n\n- **Net Asset Value (NAV)**: As previously discussed, NAV is the total value of a company's assets minus its liabilities. It is often use

In [19]:
seventh_question = ask_question("What are the metrics used to evaluate a bond?")
seventh_question

"To accurately evaluate a bond, several key metrics are typically used:\n\n- **Credit Rating**: Assessing the creditworthiness of the issuer, which indicates the risk of default. Ratings are provided by credit rating agencies like Moody's, S&P, and Fitch.\n- **Yield**: Measures the income return on the investment. This includes the yield to maturity (YTM), which is the total return anticipated on a bond if held until it matures.\n- **Coupon Rate**: The annual interest rate paid on a bond's face value by the issuer. It represents the periodic return that bondholders can expect.\n- **Maturity Date**: The date on which the bond will mature and the principal, or face value, will be paid back to the bondholders.\n- **Price**: The current market price of the bond. This can be influenced by changes in interest rates, the credit rating of the issuer, and other factors.\n- **Interest Rate Risk**: The risk of changes in bond prices due to fluctuations in interest rates. Longer-term bonds are gen

# GRADIO

In [20]:
chatbot_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a financial analyst at a stock firm. You need to answer questions related to some financial review in a  polite manner to a prospective client.

         Follow these rules:
1. Always answer based on the provided context only. Do NOT make up an answer.
2. If unsure, say "I don't have enough information to answer that"
3. Dealing with strict confidential information divulge only information asked
4. Use bullet points if needed

Context:
{context}"""),
    ("human", "Question: {question}"),
    ("ai", "Answer: "),
])


my_css = """
.gradio-container { background-color: #808080; color: blue; }
#chatbot { border: 2px solid #800000; border-radius: 15px; }
"""


def ask_question(question: str) -> str:
    # Retrieve context, explicitly setting k to limit documents
    retrieved_docs = db.similarity_search(question, k=4)

    # Format documents
    docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)

    '''# Print the retrieved context for debugging
    print("--- Retrieved Context ---")
    print(docs_content)
    print("-------------------------")'''

    # Create prompt using your custom template
    messages = chatbot_prompt.format_messages(
        question=question,
        context=docs_content
    )

    # Get LLM response
    response = llm.invoke(messages)

    return response.content


def predict(input, history=[]):
    output = ask_question(input)

    history.append((input, output))

    return history, history

# 2. Wrap it in Gradio's ChatInterface
demo = gr.ChatInterface(
    fn=ask_question,
    title="Financial advisor meeting"
)

# 3. Launch it locally (or share it)
if __name__ == "__main__":
    # Use share=True to get a public URL for 72 hours
    demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/utils.py:1201: UserWarning: Expected 1 arguments for function <function ask_question at 0x793e0ac2a2a0>, received 2.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/gradio/utils.py:1209: UserWarning: Expected maximum 1 arguments for function <function ask_question at 0x793e0ac2a2a0>, received 2.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/gradio/utils.py:1201: UserWarning: Expected 1 arguments for function <function ask_question at 0x793e210e7a60>, received 2.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/gradio/utils.py:1209: UserWarning: Expected maximum 1 arguments for function <function ask_question at 0x793e210e7a60>, received 2.
  warnings.warn(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2fae31c2efb24fe98d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Conversion to audio chatbot

In [21]:
! pip install edge-tts

In [22]:
import gradio as gr
import openai
import asyncio
import edge_tts

In [23]:
def predict_for_multimodal(message, history):
    # history is a list of dicts: [{'role': 'user', 'content': '...'}, {'role': 'assistant', 'content': '...'}]

    # Get the RAG answer using the ask_question function (defined in another cell)
    # The ask_question function expects just the question string and (optionally) history.
    rag_answer = ask_audio_question(message, history)

    # Append the new user and assistant message to history in 'messages' format
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": rag_answer})

    # Gradio Blocks with Chatbot expects history to be returned as (history, history)
    return history, history

def ask_audio_question(user_text, history):
  # Retrieve context, explicitly setting k to limit documents
    retrieved_docs = db.similarity_search(user_text, k=4)

    # Format documents
    docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)

    '''# Print the retrieved context for debugging
    print("--- Retrieved Context ---")
    print(docs_content)
    print("-------------------------")'''

    # Create prompt using your custom template
    messages = chatbot_prompt.format_messages(
        question=user_text,
        context=docs_content
    )

    # Get LLM response
    response = llm.invoke(messages)

    return response.content

async def text_to_speech(text):
    """ Converts the finance text response into a highly professional audio filee. """
    communicate = edge_tts.Communicate(text, "en-US-AndrewNeural")
    output_path = "response.mp3"
    await communicate.save(output_path)
    return "response.mp3"

def process_voice_turn(audio_path, history):
    status = "" # Initialize status message
    if audio_path is None:
        status = "No audio input received. Please record your question."
        return history, None, status

    try:
        status = "Converting audio to text..."
        # Step A: Convert voice input to text using Whisper
        client = openai.OpenAI()
        with open(audio_path, "rb") as audio_file:
            transcript = client.audio.transcriptions.create(
                model="whisper-1",
                file=audio_file
            )
        user_text = transcript.text
        status = f"User: {user_text}"

        # Step B: Pass the text to existing finance brain
        bot_text = ask_audio_question(user_text, history)
        status = "Generating audio response..."

        # Step C: Convert the bot's response to audio
        audio_output_path = asyncio.run(text_to_speech(bot_text))
        status = "Response generated successfully!"

        # Step D: Update history for Gradio's visual display with role and content
        # Each message should be a dictionary with 'role' and 'content'
        history.append({"role": "user", "content": user_text})
        history.append({"role": "assistant", "content": bot_text})

        return history, audio_output_path, status

    except Exception as e:
        status = f"An error occurred: {e}"
        return history, None, status

# Gradio Audio

In [24]:
with gr.Blocks() as demo:
    gr.Markdown("# 🎙️ Audio Conversational Chatbot")
    gr.Markdown("Record your voice, and the AI will respond back to you out loud.")

    # Keeps track of the conversation state
    history_state = gr.State([])

    with gr.Row():
        with gr.Column(scale=1):
            # Input: Microphone
            audio_input = gr.Audio(sources=["microphone"], type="filepath", label="Speak Here")
            submit_btn = gr.Button("Send", variant="primary")
            status_output = gr.Textbox(label="Status", interactive=False)

        with gr.Column(scale=2):
            # Visual Chat History
            chatbot = gr.Chatbot(label="Conversation", allow_tags=False)
            # Output: Audio Playback
            audio_output = gr.Audio(label="AI Voice Response", autoplay=True)

    # Trigger pipeline on button click
    submit_btn.click(
        fn=process_voice_turn,
        inputs=[audio_input, history_state],
        outputs=[chatbot, audio_output, status_output]
    ).then(
        # Update the hidden state history with the new chatbot history
        fn=lambda x: x,
        inputs=[chatbot],
        outputs=[history_state]
    ).then(
        # Clear the input microphone component for the next turn
        fn=lambda: None,
        outputs=[audio_input]
    )

# Launch the app
if __name__ == "__main__":
    demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://38b7d8503bda809036.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [25]:
with gr.Blocks() as demo:
    gr.Markdown("# 🎙️ Audio Conversational Chatbot")
    gr.Markdown("Record your voice, and the AI will respond back to you out loud.")

    # Keeps track of the conversation state
    history_state = gr.State([])

    with gr.Row():
        with gr.Column(scale=1):
            # Input: Microphone
            audio_input = gr.Audio(sources=["microphone"], type="filepath", label="Speak Here")
            submit_btn = gr.Button("Send", variant="primary")
            status_output = gr.Textbox(label="Status", interactive=False)

        with gr.Column(scale=2):
            # Visual Chat History
            chatbot = gr.Chatbot(label="Conversation", allow_tags=False)
            # Output: Audio Playback
            audio_output = gr.Audio(label="AI Voice Response", autoplay=True)

    # Trigger pipeline on button click
    submit_btn.click(
        fn=process_voice_turn,
        inputs=[audio_input, history_state],
        outputs=[chatbot, audio_output, status_output]
    ).then(
        # Update the hidden state history with the new chatbot history
        fn=lambda x: x,
        inputs=[chatbot],
        outputs=[history_state]
    ).then(
        # Clear the input microphone component for the next turn
        fn=lambda: None,
        outputs=[audio_input]
    )

# Launch the app
if __name__ == "__main__":
    # Explicitly set share=True for Colab environments
    demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://044dd217def82fcdd9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
